[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/01_Python%E5%9F%BA%E7%A4%8E/07_matplotlib%E5%85%A5%E9%96%80.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../ を有効化
    os.makedirs('../data', exist_ok=True)
    import numpy as np, pandas as pd
    D = '../data'; rng = np.random.default_rng(42)
    n = 120; cls = rng.choice(['A','B','C'], n)
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int)
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    days = pd.date_range('2026-06-01', periods=30)
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}
    won = np.array([rng.random()<wr[c] for c in ch])
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}
    rows = []
    for mo in months:
        for c in chs:
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:
        for c in (rng.random(size)<p): ab.append([grp,int(c)])
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 07. matplotlib入門 〜グラフを描く〜

数字の表だけでは分かりにくい特徴も、グラフにすれば一目瞭然。
**matplotlib** で、統計・マーケのノートに出てくるグラフを一通り描けるようにします。

**ゴール**：折れ線・棒・ヒストグラム・散布図・箱ひげ図・円グラフを描ける。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 日本語が□に化けるときは、環境に合わせて次の1行を有効化
# plt.rcParams['font.family'] = 'Hiragino Sans'   # Mac
# plt.rcParams['font.family'] = 'Yu Gothic'       # Windows
plt.rcParams['axes.unicode_minus'] = False

## 1. 折れ線グラフ（変化を見る）

基本は `plt.plot(x, y)` → `plt.show()`。タイトルやラベルも付けます。

In [ ]:
month = [1, 2, 3, 4, 5, 6]
temp = [22, 24, 28, 30, 27, 25]
plt.plot(month, temp, marker='o')
plt.title('気温の変化')
plt.xlabel('月'); plt.ylabel('気温(℃)')
plt.show()

## 2. 棒グラフ（大小を比べる）

In [ ]:
kyoka = ['数学', '英語', '国語', '理科']
ninzu = [8, 12, 10, 6]
plt.bar(kyoka, ninzu)
plt.title('好きな教科'); plt.ylabel('人数')
plt.show()

## 3. ヒストグラム（分布を見る）

たくさんの数値が「どのあたりに集まっているか」を見ます。統計で最重要。

In [ ]:
df = pd.read_csv('../data/students_scores.csv')
plt.hist(df['数学'], bins=10, edgecolor='white')
plt.title('数学の点数の分布'); plt.xlabel('点数'); plt.ylabel('人数')
plt.show()

## 4. 散布図（2つの量の関係）

In [ ]:
plt.scatter(df['数学'], df['英語'], alpha=0.5)
plt.title('数学と英語'); plt.xlabel('数学'); plt.ylabel('英語')
plt.show()

## 5. 箱ひげ図・円グラフ

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot([df[df['クラス']==c]['数学'] for c in ['A','B','C']],
              tick_labels=['A','B','C'])
ax[0].set_title('クラス別 数学'); ax[0].set_ylabel('点数')

df['合否'] = np.where(df['数学']>=60,'合格','不合格')
df['合否'].value_counts().plot(kind='pie', autopct='%1.0f%%', ax=ax[1])
ax[1].set_title('合否の割合'); ax[1].set_ylabel('')
plt.show()

## 6. pandasから直接プロット（近道）

`Series.plot()` / `df.plot()` を使うと、集計結果をそのままグラフ化できます。
マーケのノートでよく使う書き方です。

In [ ]:
df.groupby('クラス')['数学'].mean().plot(kind='bar', title='クラス別 数学の平均')
plt.ylabel('平均点'); plt.xticks(rotation=0)
plt.show()

---
## 🏆 練習問題

**問1.** `英語` のヒストグラム（bins=10）を描こう。

**問2.** `勉強時間`(x) と `数学`(y) の散布図を描こう。関係は見える？

**問3.** クラスごとの `英語` の平均を棒グラフにしよう（`groupby` → `.plot(kind='bar')`）。

**問4.**（仕上げ）これで道具がそろいました。`02_統計_4級/01` を開いて、
ヒストグラムと度数分布表に進みましょう！

In [ ]:
# 問1〜3


<details><summary>解答例</summary>

```python
plt.hist(df['英語'], bins=10, edgecolor='white'); plt.show()
plt.scatter(df['勉強時間'], df['数学'], alpha=0.5); plt.show()
df.groupby('クラス')['英語'].mean().plot(kind='bar'); plt.show()
```
</details>

🎉 **データ分析の三種の神器（NumPy・pandas・matplotlib）クリア！**
これで統計・マーケのノートをスムーズに読めます。